# 20 — Uji fungsi, ketahanan, waktu respons, dan pemeriksaan manusia

Lanjutan notebook 18–19; acuan Bab III §3.2.5(c–d). Pengujian endpoint memakai aplikasi yang berjalan,
bukan mock. Ketahanan mengadaptasi **minimum functionality** dan **invariance testing** dari
[Ribeiro dkk. (2020), CheckList](https://aclanthology.org/2020.acl-main.442/).
Pertanyaan faktual/interpretatif menguji kemampuan dasar; parafrasa dan typo diuji berpasangan dengan
pertanyaan asli. OOS (di luar korpus) dinilai dari penolakan yang tepat tanpa fakta karangan, bukan token F1.

Waktu adalah pengukuran operasional sistem, bukan metrik yang diklaim berasal dari CheckList.
Catat perangkat, kondisi pemuatan model, jumlah ulangan, dan panjang keluaran.
Tidak menyimpulkan TTFT atau token/detik dari durasi total; stream aplikasi hanya mengirim status tahapan.

## Cara menjalankan (lokal, bukan Kaggle)

1. Unduh folder hasil 18 dari Kaggle; letakkan di `Hasil/tujuan3/pilot_v2` pada proyek ini.
2. Pada terminal folder **Sistem QA RAG**, aktifkan lingkungan Python aplikasi, pastikan `.env`
   dan empat model sudah tersedia. Jalankan `ollama serve` bila layanan belum aktif.
   Untuk setup pertama, ikuti README aplikasi (`pip install -r requirements.txt`,
   `python -m deploy.setup_ollama`).
3. Jalankan `uvicorn app.main:app --reload` untuk pemeriksaan dev. Untuk pengukuran waktu
   final, restart tanpa `--reload`: `uvicorn app.main:app --host 127.0.0.1 --port 8000`.
4. Buka notebook ini di Jupyter/VS Code lokal. Kernel notebook cukup mempunyai pandas,
   numpy, requests; krippendorff hanya dibutuhkan ketika rekap penilaian manusia.
5. Sesuaikan RUN_NAME/API_URL, jalankan preflight, lalu aktifkan flag pengujian satu per satu.

Notebook mengirim HTTP ke aplikasi; **aplikasi tidak menjalankan ipynb**. Tidak perlu OpenAI
untuk 20. `localhost` Kaggle menunjuk mesin Kaggle, bukan laptop kamu; jangan memindahkan
20 ke Kaggle dengan URL localhost yang sama. Jangan jalankan benchmarking bersama eksperimen lain.
Hasil baru uji lokal disimpan terpisah dalam `local_tests/<LOCAL_RUN_NAME>`.

In [ ]:
from pathlib import Path
import os, json, hashlib, sys, platform, time, random
import importlib.metadata as metadata
import pandas as pd
import numpy as np

# Temukan akar proyek dari lokasi kernel notebook.
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "Hasil/tujuan2/test.csv").exists():
            return p
    raise FileNotFoundError("Jalankan kernel dari folder SkripsiNabil atau tujuan3.")

# Hitung identitas isi artefak untuk mencegah pencampuran eksperimen.
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

# Simpan JSON secara atomik agar berkas tidak setengah tertulis.
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)

ROOT = find_root()
RUN_NAME = "pilot_v2"  # Ganti menjadi main_v1 untuk pengujian penuh.
OUT = ROOT / "Hasil/tujuan3" / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42

SOURCE_OUT = OUT
LOCAL_RUN_NAME = "dev_v1"  # Final gunakan nama baru, misalnya final_v1.
OUT = SOURCE_OUT / "local_tests" / LOCAL_RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
import requests, re
API_URL = os.environ.get("RAG_API_URL", "http://127.0.0.1:8000")
OLLAMA_URL = os.environ.get("INFERENCE_URL", "http://127.0.0.1:11434")
RUN_API_TESTS = False
RUN_TIMING = False
RUN_ROBUSTNESS = False
SERVER_HARDWARE = ""  # Wajib isi CPU, RAM, GPU/VRAM server sebelum timing.
SERVER_RUN_MODE = "dev_reload"  # Ganti "no_reload" untuk timing final.
TIMING_REPEATS = 3  # Keputusan operasional; bekukan sebelum pengujian final.
TIMING_QUESTIONS = 3
MODEL_KEYS = ["gemma2-base", "gemma2-finetuned", "llama3.2-base", "llama3.2-finetuned"]
selected = pd.read_csv(SOURCE_OUT / "test_selected.csv", dtype=str, keep_default_na=False)
# Data RAGAS boleh tanpa acuan; bukti untuk penilaian manusia tetap harus diperiksa.
for column in ["ans_ref", "context_joined"]:
    if column not in selected:
        selected[column] = ""

# Verifikasi koneksi dan empat model sebelum mengirim permintaan pengujian.
def preflight():
    response = requests.get(API_URL + "/openapi.json", timeout=15)
    response.raise_for_status()
    assert "/api/ask" in response.json()["paths"], "Endpoint aplikasi tidak sesuai."
    response = requests.get(OLLAMA_URL + "/api/tags", timeout=15)
    response.raise_for_status()
    available = {m["name"].removesuffix(":latest") for m in response.json()["models"]}
    assert set(MODEL_KEYS) <= available, "Model belum tersedia: " + str(set(MODEL_KEYS) - available)
    print("Endpoint dan empat model tersedia; belum berarti seluruh fungsi aplikasi lulus.")

if RUN_API_TESTS or RUN_TIMING or RUN_ROBUSTNESS:
    preflight()

# Kirim satu permintaan dan ukur waktu end-to-end di sisi klien.
def ask_api(question, model_key, use_rag=True):
    start = time.perf_counter()
    response = requests.post(API_URL + "/api/ask", json={"question": question,
               "model_key": model_key, "use_rag": use_rag, "top_k": 5}, timeout=(10, 900))
    elapsed = time.perf_counter() - start
    response.raise_for_status()
    return response.json(), elapsed

# Simpan hasil uji beserta keluaran aktual tanpa mengasumsikan semua kasus lulus.
def functional_tests():
    rows = []
    for case, question, model in [("empty", "", MODEL_KEYS[0]), ("whitespace", "   ", MODEL_KEYS[0]),
                                   ("unknown_model", "Apa itu BPS?", "MODEL_TIDAK_ADA")]:
        try:
            r = requests.post(API_URL + "/api/ask", json={"question": question, "model_key": model}, timeout=30)
            rows.append({"case": case, "status": "pass" if r.status_code in [400, 422] else "fail",
                         "actual": str(r.status_code) + " " + r.text[:1000]})
        except requests.RequestException as exc:
            rows.append({"case": case, "status": "error", "actual": str(exc)})
    for model in MODEL_KEYS:
        for rag in [False, True]:
            case = f"response_schema__{model}__rag_{rag}"
            try:
                data, elapsed = ask_api(selected.iloc[0].instruction, model, rag)
                valid = isinstance(data.get("answer"), str) and bool(data["answer"].strip())
                valid = valid and isinstance(data.get("sources"), list)
                valid = valid and data.get("config") == {"model_key": model, "use_rag": rag}
                if not rag:
                    valid = valid and data["sources"] == [] and data.get("process", {}).get("expanded_queries") == []
                sources = data.get("sources", [])
                citations = [int(x) for x in re.findall(r"\[(\d+)\]", data.get("answer", ""))]
                citation_valid = all(1 <= n <= len(sources) for n in citations)
                rows.append({"case": case, "status": "pass" if valid and citation_valid else "fail",
                             "actual": json.dumps(data, ensure_ascii=False), "client_s": elapsed,
                             "note": "Hanya skema/nomor sitasi; dukungan fakta wajib diperiksa manual."})
            except Exception as exc:
                rows.append({"case": case, "status": "error", "actual": str(exc)})
    # Stream diperiksa sebagai event status, bukan token streaming.
    try:
        with requests.post(API_URL + "/api/ask/stream", json={"question": selected.iloc[0].instruction,
                  "model_key": MODEL_KEYS[0], "use_rag": True, "top_k": 5}, stream=True, timeout=(10, 900)) as r:
            r.raise_for_status()
            lines = list(r.iter_lines(decode_unicode=True))
        events = [s[7:].strip() for s in lines if s.startswith("event: ")]
        expected = ["expanding", "retrieving", "reranking", "reranked", "citing", "generating", "done"]
        rows.append({"case": "stream_stage_order", "status": "pass" if events == expected else "fail",
                     "actual": json.dumps(events)})
    except Exception as exc:
        rows.append({"case": "stream_stage_order", "status": "error", "actual": str(exc)})
    return pd.DataFrame(rows)

if RUN_API_TESTS:
    functional = functional_tests()
    functional.to_csv(OUT / "functional_results.csv", index=False)
    display(functional[["case", "status"]])
else:
    print("Uji endpoint belum dijalankan.")

## 1. Kasus yang memerlukan pemeriksaan terarah

Status `not_run` tidak berarti lulus. Isikan keluaran aktual dan bukti setelah menjalankan langkah berikut
pada lingkungan uji. Kasus gangguan layanan jangan dijalankan pada server yang sedang dipakai pengguna lain.
Pengujian referensi harus membaca publikasi, bukan hanya memastikan URL ada.


In [ ]:
# Buat lembar kasus manual tanpa menimpa hasil pemeriksaan sebelumnya.
def prepare_manual_cases():
    cases = [
        ("no_source", "Pada lingkungan uji, atur threshold reranker sangat tinggi lalu restart aplikasi; kirim pertanyaan.",
         "sources kosong; rekam apakah jawaban menyatakan keterbatasan tanpa fakta karangan; pulihkan threshold."),
        ("inference_down", "Hentikan Ollama pada lingkungan uji; kirim pertanyaan RAG nonaktif, lalu aktif; nyalakan kembali.",
         "Pesan kegagalan jelas pada kedua mode dan permintaan baru berhasil setelah layanan pulih."),
        ("source_support", "Buka bps_url dan halaman tiap sumber untuk sampel jawaban RAG.",
         "Judul/bagian/halaman sesuai; klaim jawaban benar-benar didukung sumber."),
        ("sequential_isolation", "Kirim A, B yang berbeda topik, lalu A lagi; simpan ketiga respons.",
         "Tidak ada konteks/jawaban B yang terbawa ke A atau sebaliknya; perbedaan stokastik dicatat."),
        ("ui_stream", "Kirim pertanyaan di browser dan amati status hingga jawaban akhir.",
         "Status dan jawaban dapat ditampilkan tanpa error UI.")]
    return pd.DataFrame([{"case": c, "steps": s, "expected": e, "status": "not_run",
                          "actual": "", "evidence_path": "", "reviewer": ""} for c,s,e in cases])

if not (OUT / "functional_manual.csv").exists():
    prepare_manual_cases().to_csv(OUT / "functional_manual.csv", index=False)


## 2. Waktu respons dengan pemuatan model dan setelah model siap

Sebelum setiap blok, model yang diuji dikeluarkan dari memori Ollama dan keberhasilannya diverifikasi lewat
`/api/ps`. Permintaan pertama diberi label `model_unloaded`, bukan cold-start seluruh sistem:
embedding/reranker dan cache sistem operasi mungkin sudah hangat. Permintaan berikutnya diberi label
`warm`; status model diverifikasi sebelum permintaan. Jangan jalankan judge RAGAS bersamaan.

Waktu retrieval mencakup ekspansi kueri; pemuatan model pada mode RAG dapat terjadi dalam komponen tersebut.
Laporkan median dan p95, dengan catatan p95 dari sedikit ulangan belum stabil. Durasi klien mencakup jaringan;
durasi komponen berasal dari endpoint aplikasi. Catat spesifikasi **server** secara manual jika endpoint remote.


In [ ]:
# Periksa model yang sedang termuat menurut runtime Ollama.
def loaded_models():
    r = requests.get(OLLAMA_URL + "/api/ps", timeout=15)
    r.raise_for_status()
    return {m.get("name", m.get("model", "")).removesuffix(":latest") for m in r.json()["models"]}

# Keluarkan model dari memori dan pastikan runtime telah melepasnya.
def unload_model(model):
    r = requests.post(OLLAMA_URL + "/api/generate", json={"model": model, "keep_alive": 0}, timeout=60)
    r.raise_for_status()
    for _ in range(20):
        if model not in loaded_models():
            return
        time.sleep(0.5)
    raise RuntimeError("Model belum dilepas; kondisi model_unloaded tidak sah.")

# Ukur blok waktu untuk pertanyaan yang sama pada semua kondisi model dan RAG.
def benchmark_latency():
    assert SERVER_HARDWARE.strip(), "Isi spesifikasi server aktual."
    assert SERVER_RUN_MODE == "no_reload", "Matikan auto-reload untuk pengukuran waktu."
    rows = []
    sample = selected.sample(n=min(TIMING_QUESTIONS, len(selected)), random_state=SEED)
    blocks = [(m, rag, ex) for m in MODEL_KEYS for rag in [False, True] for ex in sample.to_dict("records")]
    random.Random(SEED).shuffle(blocks)
    timing_id = time.strftime("%Y%m%d_%H%M%S")
    folder = OUT / "timing" / timing_id
    folder.mkdir(parents=True, exist_ok=False)
    for model, rag, ex in blocks:
        # Bersihkan keempat model uji agar residensi model lain tidak mengacaukan warm/cold.
        for name in MODEL_KEYS:
            if name in loaded_models():
                unload_model(name)
        for repeat in range(TIMING_REPEATS + 1):
            state = "model_unloaded" if repeat == 0 else "warm"
            row = {"model_key": model, "use_rag": rag, "record_id": ex["record_id"],
                   "repeat": repeat, "load_state": state}
            try:
                if repeat > 0 and model not in loaded_models():
                    raise RuntimeError("Model terlepas sebelum warm request; ukur ulang blok.")
                data, elapsed = ask_api(ex["instruction"], model, rag)
                row.update(status="ok", client_total_s=elapsed, **data["latency"],
                           output_chars=len(data["answer"]), source_count=len(data["sources"]))
                write_json(folder / f"{model}_{rag}_{ex['record_id']}_{repeat}.json", {"measurement": row, "response": data})
            except Exception as exc:
                row.update(status="error", error=str(exc))
            rows.append(row)
            pd.DataFrame(rows).to_csv(folder / "latency_detail.csv", index=False)
    frame = pd.DataFrame(rows)
    valid = frame[frame.status.eq("ok")]
    if not valid.empty:
        columns = ["client_total_s", "total_s", "retrieval_s", "citation_s", "generation_s", "output_chars"]
        valid.groupby(["model_key", "use_rag", "load_state"])[columns].agg(
            ["count", "mean", "median", lambda x: x.quantile(.95)]).to_csv(folder / "latency_summary.csv")
    write_json(folder / "timing_environment.json", {"client_platform": platform.platform(),
               "api_url": API_URL, "ollama_url": OLLAMA_URL, "repeats_warm": TIMING_REPEATS,
               "question_ids": sample.record_id.tolist(), "server_hardware": SERVER_HARDWARE, "server_run_mode": SERVER_RUN_MODE,
               "note": "Model unloaded, bukan cold-start seluruh sistem; tidak mengukur TTFT."})
    return frame

if RUN_TIMING:
    timing = benchmark_latency()
    display(timing)

## 3. Susun dan sahkan pasangan uji ketahanan

Lembar awal berisi pertanyaan asli, kandidat typo, dan slot parafrasa. Kandidat typo menukar huruf pada satu
kata, tetapi **tetap harus direview** agar maksud tidak berubah. Tambahkan OOS setelah memeriksa cakupan korpus;
tidak cukup sekadar mengambil pertanyaan yang tidak dijumpai dalam dataset QA.
Untuk setiap variasi, pertahankan `parent_id`, `ans_ref`, dan konteks bukti asli. Isi `approved=1` dan reviewer.
Kategori factual/interpretative dapat dilihat dari `subtask_type` di test; pastikan keduanya terwakili.


In [ ]:
# Buat satu typo ringan dengan menukar dua huruf di dalam kata panjang.
def typo_variant(question):
    match = re.search(r"\b[A-Za-z]{6,}\b", question)
    if not match:
        return ""
    word = match.group()
    variant = word[:2] + word[3] + word[2] + word[4:]
    return question[:match.start()] + variant + question[match.end():]

# Siapkan pasangan asli-variasi; belum dianggap label yang disahkan manusia.
def prepare_robustness():
    rows = []
    for ex in selected.to_dict("records"):
        for kind, question in [("original", ex["instruction"]), ("typo", typo_variant(ex["instruction"])), ("paraphrase", "")]:
            rows.append({"item_id": ex["record_id"] + "__" + kind, "parent_id": ex["record_id"],
                         "kind": kind, "question": question, "ans_ref": ex["ans_ref"],
                         "subtask_type": ex.get("subtask_type", ""), "evidence": ex["context_joined"],
                         "approved": 0, "reviewer": "", "notes": ""})
    rows.append({"item_id": "oos_001", "parent_id": "", "kind": "out_of_scope", "question": "",
                 "ans_ref": "", "subtask_type": "out_of_scope", "evidence": "",
                 "approved": 0, "reviewer": "", "notes": "Isi setelah memeriksa cakupan korpus."})
    return pd.DataFrame(rows)

if not (OUT / "robustness_review.csv").exists():
    prepare_robustness().to_csv(OUT / "robustness_review.csv", index=False)

# Jalankan semua item yang disahkan pada delapan kondisi aplikasi.
def run_robustness():
    items = pd.read_csv(OUT / "robustness_review.csv", dtype=str, keep_default_na=False)
    items = items[items.approved.eq("1")].copy()
    assert not items.empty, "Belum ada item robustness disahkan."
    assert items.item_id.is_unique and items.question.str.strip().ne("").all()
    assert items.reviewer.str.strip().ne("").all()
    assert items.kind.isin(["original", "typo", "paraphrase", "out_of_scope"]).all()
    originals = items[items.kind.eq("original")].set_index("parent_id")
    for item in items[items.kind.isin(["typo", "paraphrase"])].itertuples():
        assert item.parent_id in originals.index, "Sahkan pertanyaan original pasangannya juga."
        assert item.ans_ref == originals.loc[item.parent_id, "ans_ref"], "Acuan pasangan berubah."
    folder = OUT / "robustness" / digest(items.to_dict("records"))[:16]
    folder.mkdir(parents=True, exist_ok=True)
    rows = []
    for ex in items.to_dict("records"):
        for model in MODEL_KEYS:
            for rag in [False, True]:
                path = folder / (digest([ex["item_id"], model, rag]) + ".json")
                row = json.loads(path.read_text(encoding="utf-8")) if path.exists() else None
                if row is None or row["status"] != "ok":
                    row = {**ex, "model_key": model, "use_rag": rag}
                    try:
                        data, elapsed = ask_api(ex["question"], model, rag)
                        row.update(status="ok", answer=data["answer"], response=data, client_s=elapsed)
                    except Exception as exc:
                        row.update(status="error", error=str(exc))
                    write_json(path, row)
                rows.append(row)
    results = pd.DataFrame(rows)
    results.to_json(folder / "results.jsonl", orient="records", lines=True, force_ascii=False)
    review = results[results.status.eq("ok")].copy()
    for column in ["meaning_preserved", "factually_correct", "appropriate_refusal", "unsupported_claim", "human_notes"]:
        review[column] = ""
    if not (folder / "human_review.csv").exists():
        review.drop(columns=["response"], errors="ignore").to_csv(folder / "human_review.csv", index=False)
    print("Review hasil di", folder)
    return results

if RUN_ROBUSTNESS:
    robustness = run_robustness()


## 4. Cakupan uji robustness dan pemeriksaan manusia

Rekap otomatis berikut hanya menunjukkan keberhasilan permintaan, bukan skor kualitas jawaban.
Token F1 tidak dihitung dan tidak ada impor fungsi notebook 18/19. Nilai ketahanan faktual
melalui human_review.csv: meaning_preserved, factually_correct, appropriate_refusal,
unsupported_claim (0/1). Bandingkan original dan variasi pada model serta mode yang sama.


In [ ]:
# Laporkan keberhasilan/gagal permintaan tanpa menganggapnya skor kualitas jawaban.
def summarize_robustness(results):
    return results.groupby(["kind", "model_key", "use_rag", "status"], dropna=False).size().rename("n_requests").reset_index()

# Ringkas penolakan OOS hanya dari penilaian manusia yang benar-benar terisi.
def summarize_oos(review_path):
    frame = pd.read_csv(review_path, dtype=str, keep_default_na=False)
    frame = frame[frame.kind.eq("out_of_scope")].copy()
    for c in ["appropriate_refusal", "unsupported_claim"]:
        assert frame[c].isin(["", "0", "1"]).all(), f"{c} harus 0/1 atau kosong."
        frame[c] = pd.to_numeric(frame[c], errors="coerce")
    return frame.groupby(["model_key", "use_rag"])[["appropriate_refusal", "unsupported_claim"]].agg(["mean", "count"])

if RUN_ROBUSTNESS:
    display(summarize_robustness(robustness))
# Setelah human_review.csv diisi: display(summarize_oos(Path("lokasi/human_review.csv")))

## 5. Paket evaluasi manusia sebagai pelengkap

Adaptasi rancangan proposal: 1% pertanyaan (dibulatkan ke atas, minimum satu), seed 42, lima evaluator,
skala 1–5 untuk fluency, factual correctness, dan completeness. Sampling pada pertanyaan, lalu **delapan
kondisi lengkap** ikut dinilai agar tetap berpasangan. Identitas model disembunyikan dan urutan diacak.
Kunci pemetaan disimpan terpisah; jangan dibagikan kepada evaluator.
Dasar pelaporan instrumen: [van der Lee dkk. (2021)](https://doi.org/10.1016/j.csl.2020.101151).
Sampel 1% merupakan kebijakan proposal, bukan jaminan kecukupan statistik. Penilai tetap dapat menandai
acuan sintetis meragukan. Hasil penilaian manusia baru tersedia setelah lembar benar-benar diisi.


In [ ]:
# Bentuk paket buta dengan konteks bukti dan keluaran delapan kondisi yang lengkap.
def prepare_human_evaluation():
    paths = sorted((SOURCE_OUT / "predictions").glob("*.json"))
    rows = [json.loads(p.read_text(encoding="utf-8")) for p in paths]
    df = pd.DataFrame(rows)
    df = df[df.status.eq("ok") & df.protocol.eq("end_to_end")].copy()
    counts = df.groupby("record_id").size()
    complete_ids = counts[counts.eq(8)].index
    n = max(1, int(np.ceil(.01 * len(selected))))
    if len(complete_ids) < n:
        raise ValueError("Pertanyaan dengan delapan kondisi lengkap belum cukup.")
    sampled = pd.Series(sorted(complete_ids)).sample(n=n, random_state=SEED).tolist()
    subset = df[df.record_id.isin(sampled)].sample(frac=1, random_state=SEED).copy()
    subset["eval_id"] = [f"E{i:05d}" for i in range(1, len(subset)+1)]
    folder = OUT / "human_evaluation"
    if folder.exists():
        raise FileExistsError("Paket sudah ada; jangan menimpa penilaian. Gunakan folder run baru.")
    folder.mkdir()
    subset[["eval_id", "record_id", "model_key", "condition", "protocol"]].to_csv(folder / "PRIVATE_key.csv", index=False)
    evidence = selected.set_index("record_id").context_joined.to_dict()
    if "ans_ref" not in subset:
        subset["ans_ref"] = ""
    packet = subset[["eval_id", "question", "ans_ref", "answer"]].copy()
    packet["evidence_reference"] = subset.record_id.map(evidence)
    packet["retrieved_context"] = subset.contexts.map(lambda x: "\n\n".join(x))
    for col in ["fluency", "factual_correctness", "completeness", "ans_ref_meragukan", "notes"]:
        packet[col] = ""
    for evaluator in range(1, 6):
        packet.sample(frac=1, random_state=SEED+evaluator).to_csv(folder / f"evaluator_{evaluator}.csv", index=False)
    rubric = """Gunakan skala 1–5. Jangan menilai berdasarkan gaya yang disukai saja.
Fluency: 5 lancar; 4 kesalahan kecil; 3 beberapa gangguan; 2 sulit dipahami; 1 tidak dapat dipahami.
Factual correctness: 5 semua klaim didukung; 4 fakta utama benar dengan kesalahan minor;
3 sebagian benar; 2 sebagian besar keliru; 1 tidak ada klaim yang dapat dibenarkan.
Completeness: 5 semua kebutuhan terjawab; 4 hampir semua; 3 sebagian informasi utama;
2 sedikit informasi relevan; 1 tidak menjawab.
Baca sumber publikasi bila bukti belum cukup. Acuan sintetis dapat salah: tandai ans_ref_meragukan 1/0
dan jelaskan notes. Konteks retrieval kosong tidak otomatis berarti ketepatan faktual bernilai 1.
Kalibrasikan interpretasi rubrik bersama pada contoh terpisah sebelum penilaian utama.
"""
    (folder / "PETUNJUK.txt").write_text(rubric, encoding="utf-8")
    return folder

# Jalankan sesudah semua kondisi selesai; tidak mengirim paket kepada orang lain.
# prepare_human_evaluation()

## 6. Rekap penilaian manusia dan kesepakatan antarpenilai

Jalankan setelah kelima lembar selesai. Skor ordinal diringkas per dimensi/kondisi; Krippendorff's alpha
dihitung per dimensi dengan unit = eval_id dan penilai = evaluator. Alpha memakai jarak ordinal,
bukan korelasi antarskor. Acuan metode: [Krippendorff (2011), Computing Krippendorff's Alpha-Reliability](https://www.asc.upenn.edu/krippendorffs-alpha-reliability).
Kasus acuan meragukan dicatat untuk adjudikasi, tidak dihapus otomatis.


In [ ]:
# Pasang sekali di kernel evaluasi bila diperlukan:
# %pip install "krippendorff==0.8.1"

# Validasi lima lembar lalu hitung ringkasan skor dan alpha ordinal.
def aggregate_human_evaluation():
    import krippendorff
    folder = OUT / "human_evaluation"
    key = pd.read_csv(folder / "PRIVATE_key.csv", dtype=str)
    dimensions = ["fluency", "factual_correctness", "completeness"]
    frames = []
    for evaluator in range(1, 6):
        frame = pd.read_csv(folder / f"evaluator_{evaluator}.csv", dtype=str, keep_default_na=False)
        assert frame.eval_id.is_unique and set(frame.eval_id) == set(key.eval_id)
        for dim in dimensions:
            assert frame[dim].isin(["1", "2", "3", "4", "5"]).all(), f"Lembar {evaluator}: {dim} belum lengkap."
            frame[dim] = frame[dim].astype(int)
        assert frame.ans_ref_meragukan.isin(["0", "1"]).all(), "Isi penanda acuan meragukan 0/1."
        frame["evaluator"] = evaluator
        frames.append(frame)
    combined = pd.concat(frames).merge(key, on="eval_id", validate="many_to_one")
    combined.groupby(["model_key", "condition"])[dimensions].agg(["mean", "median", "count"]).to_csv(folder / "human_summary.csv")
    reliability = []
    for dim in dimensions:
        matrix = combined.pivot(index="evaluator", columns="eval_id", values=dim).to_numpy(dtype=float)
        try:
            alpha = float(krippendorff.alpha(reliability_data=matrix, level_of_measurement="ordinal", value_domain=[1,2,3,4,5]))
            status = "ok" if np.isfinite(alpha) else "undefined_no_variation"
        except ValueError as exc:
            alpha, status = np.nan, str(exc)
        reliability.append({"dimension": dim, "alpha_ordinal": alpha, "status": status,
                            "n_evaluators": matrix.shape[0], "n_items": matrix.shape[1]})
    result = pd.DataFrame(reliability)
    result.to_csv(folder / "krippendorff_alpha.csv", index=False)
    combined[combined.ans_ref_meragukan.eq("1")].to_csv(folder / "reference_adjudication.csv", index=False)
    return result

# aggregate_human_evaluation()
